# 07 · Verificación Rcamiz ↔ scikit-learn ↔ autoencoder lineal
**Tesis:** Medición del ciclo financiero en Perú mediante técnicas de reducción dimensional y machine learning
**Autor:** Roberto Samaniego Salcedo · **Asesor:** Dr. Sergio Camiz

---

**Objetivo:** comprobar que el PCA de la pipeline en Python (scikit-learn) reproduce el PCA del
paquete Rcamiz del asesor sobre el mismo dataset, y cerrar numéricamente la cadena
**Rcamiz → scikit-learn → autoencoder lineal** (el último tramo está verificado en `08`).

**Entrada:** `data/processed/dataset_transformado_mensual.csv` (227 meses × 17 variables) y las
salidas de `notebooks/07_pca_rcamiz.R` (`data/results/07_rcamiz_*.csv`).

**Criterios de equivalencia:**

1. Mismos valores propios e inercia por dimensión.
2. Mismas cargas (correlaciones variable–componente) tras aplicar la regla de signo de Rcamiz.
3. Mismos scores de los meses.

Dos convenciones deben alinearse antes de comparar: Rcamiz calcula la varianza con divisor
$n$ (y scikit-learn reporta `explained_variance_` con divisor $n-1$), y Rcamiz orienta cada
eje de modo que la suma de cuadrados de las cargas positivas supere a la de las negativas.

In [1]:
import os
# Ejecutar desde la raíz del repositorio
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X = pd.read_csv('data/processed/dataset_transformado_mensual.csv', index_col=0)
n, p = X.shape
print(f'Dataset: {n} meses x {p} variables, {X.index[0]} a {X.index[-1]}')

# Control: no se trabaja con datos incompletos
assert not X.isna().any().any(), 'Hay valores faltantes'
calendario = pd.date_range(X.index[0], X.index[-1], freq='MS')
assert len(calendario) == n, 'El calendario mensual tiene huecos'
print('Calendario mensual completo, sin valores faltantes')

Dataset: 227 meses x 17 variables, 2007-02-01 a 2025-12-01
Calendario mensual completo, sin valores faltantes


## 1. PCA con scikit-learn, alineado a las convenciones de Rcamiz

In [2]:
# Estandarización con desviación poblacional (divisor n), igual que Rcamiz
Z = StandardScaler().fit_transform(X)
modelo = PCA().fit(Z)

# Valores propios con divisor n (scikit-learn usa n - 1)
valores_propios = modelo.explained_variance_ * (n - 1) / n
vectores = modelo.components_.T.copy()

# Regla de signo de Rcamiz: si la suma de cuadrados de los coeficientes negativos
# supera a la de los positivos, se invierte el eje
margen_signo = []
for k in range(p):
    v = vectores[:, k]
    neg, pos = (v[v < 0] ** 2).sum(), (v[v > 0] ** 2).sum()
    if neg > pos:
        vectores[:, k] = -v
        neg, pos = pos, neg
    margen_signo.append((neg, pos))

cargas_sk = vectores * np.sqrt(valores_propios)   # correlaciones variable-componente
scores_sk = Z @ vectores                          # coordenadas de los meses

## 2. Comparación con las salidas de Rcamiz

In [3]:
vp_rc = pd.read_csv('data/results/07_rcamiz_valores_propios.csv', index_col=0)
cargas_rc = pd.read_csv('data/results/07_rcamiz_cargas.csv', index_col=0)
scores_rc = pd.read_csv('data/results/07_rcamiz_scores.csv', index_col=0)

# Mismo orden de variables y de meses
assert list(cargas_rc.index) == list(X.columns)
assert list(scores_rc.index) == list(X.index)

comparacion = pd.DataFrame({
    'valor_propio_rcamiz': vp_rc['eigen'].values,
    'valor_propio_sklearn': valores_propios,
    'inercia_acum_rcamiz_%': vp_rc['cum.in'].values,
    'inercia_acum_sklearn_%': np.cumsum(valores_propios) / valores_propios.sum() * 100,
    'dif_max_cargas': np.abs(cargas_sk - cargas_rc.values).max(axis=0),
    'dif_max_scores': np.abs(scores_sk - scores_rc.values).max(axis=0),
    'abs_r_scores': [abs(np.corrcoef(scores_sk[:, j], scores_rc.values[:, j])[0, 1]) for j in range(p)],
}, index=[f'Dim_{j + 1}' for j in range(p)])

pd.set_option('display.float_format', lambda x: f'{x:.6g}')
comparacion.head(5)

,valor_propio_rcamiz,valor_propio_sklearn,inercia_acum_rcamiz_%,inercia_acum_sklearn_%,dif_max_cargas,dif_max_scores,abs_r_scores
Dim_1,3.95646,3.95646,23.2733,23.2733,2.98025e-15,2.35367e-14,1
Dim_2,2.71152,2.71152,39.2234,39.2234,1.88738e-15,1.46549e-14,1
Dim_3,1.89576,1.89576,50.3749,50.3749,7.77156e-16,1.15463e-14,1
Dim_4,1.46632,1.46632,59.0004,59.0004,1.72085e-15,1.28786e-14,1
Dim_5,1.03302,1.03302,65.077,65.077,2.55906e-14,2.09055e-13,1


In [4]:
print('Diferencia máxima en valores propios:', np.abs(valores_propios - vp_rc['eigen'].values).max())
print('Diferencia máxima en cargas (17 dimensiones):', np.abs(cargas_sk - cargas_rc.values).max())
print('Diferencia máxima en scores (17 dimensiones):', np.abs(scores_sk - scores_rc.values).max())

Diferencia máxima en valores propios: 4.679678866637005e-11
Diferencia máxima en cargas (17 dimensiones): 3.123890035539034e-14
Diferencia máxima en scores (17 dimensiones): 2.0905499553691698e-13


## 3. Cierre de la cadena con el autoencoder lineal (`08`)

In [5]:
anclaje = pd.read_csv('data/results/08_ae_anclaje_lineal.csv')
cadena = pd.DataFrame({
    'd': anclaje['d'],
    'inercia_rcamiz_%': vp_rc['cum.in'].values[:len(anclaje)],
    'inercia_sklearn_%': (np.cumsum(valores_propios) / valores_propios.sum() * 100)[:len(anclaje)],
    'inercia_pca_08_%': anclaje['inercia_pca_%'],
    'mse_pca_08': anclaje['mse_pca'],
    'mse_ae_lineal_08': anclaje['mse_ae_lineal'],
})
cadena

,d,inercia_rcamiz_%,inercia_sklearn_%,inercia_pca_08_%,mse_pca_08,mse_ae_lineal_08
0,1,23.2733,23.2733,23.2733,0.767267,0.767267
1,2,39.2234,39.2234,39.2234,0.607766,0.607766
2,3,50.3749,50.3749,50.3749,0.496251,0.496251
3,4,59.0004,59.0004,59.0004,0.409996,0.409996
4,5,65.077,65.077,65.077,0.34923,0.34923


## 4. Margen de la regla de signo

In [6]:
signo = pd.DataFrame(margen_signo, columns=['suma_cuad_negativos', 'suma_cuad_positivos'],
                     index=[f'Dim_{j + 1}' for j in range(p)])
signo['margen'] = signo['suma_cuad_positivos'] - signo['suma_cuad_negativos']
signo.head(5)

,suma_cuad_negativos,suma_cuad_positivos,margen
Dim_1,0.494812,0.505188,0.0103752
Dim_2,0.0510196,0.94898,0.897961
Dim_3,0.193066,0.806934,0.613868
Dim_4,0.486519,0.513481,0.0269625
Dim_5,0.303601,0.696399,0.392797


In [7]:
# Guardar la tabla de comparación
comparacion.to_csv('data/results/07_verificacion_rcamiz_sklearn.csv')
signo.to_csv('data/results/07_margen_signo.csv')
print('Guardado: data/results/07_verificacion_rcamiz_sklearn.csv y 07_margen_signo.csv')

Guardado: data/results/07_verificacion_rcamiz_sklearn.csv y 07_margen_signo.csv


## 5. Lectura

1. **Equivalencia exacta.** Con las convenciones alineadas (divisor $n$ y regla de signo de
   Rcamiz), scikit-learn reproduce el PCA de Rcamiz en las 17 dimensiones: valores propios,
   cargas y scores coinciden hasta el error de redondeo de punto flotante (diferencias del
   orden de $10^{-11}$ o menores) y $|r| = 1$ entre scores.
2. **Cadena cerrada.** La inercia acumulada coincide en Rcamiz, scikit-learn y `08`
   (50,37 % con d = 3; 59,00 % con d = 4), y el autoencoder lineal alcanza el mismo MSE que el
   PCA para d = 1…5 (`08`). La cadena Rcamiz → scikit-learn → autoencoder lineal queda
   verificada sobre el dataset de 227 meses.
3. **Orientación frágil en Dim_1 y Dim_4.** La regla de signo decide con márgenes de 0,01 y
   0,03 en la suma de cuadrados: un cambio pequeño en los datos (o en la ventana, en EPCA)
   puede invertir esos ejes. La interpretación de su signo debe reportarse junto con este
   margen, y en EPCA conviene complementar la regla con la de continuidad entre ventanas.